# One-Stage Detection: FCOS with torchvision

Fine-tune an anchor-free, one-stage detector ([FCOS](https://arxiv.org/abs/1904.01355)) on PASCAL VOC 2007.

The model, target assignment, losses and post-processing all come from `torchvision.models.detection.FCOS`;
no forward or backward pass is written by hand. A few pieces are still implemented in a few lines, because
a CV engineer should be able to write them from memory and debug them when they go wrong: the FCOS location
grid with LTRB box encoding, the centerness target, focal loss, and NMS. Each one is checked against torchvision.

| Original assignment (hand-written) | This notebook |
|---|---|
| `VOC2007DetectionTiny`, padded `(B, 40, 5)` box tensor | `torchvision.datasets.VOCDetection` + `transforms.v2` + `tv_tensors.BoundingBoxes` |
| `DetectorBackboneWithFPN` | `fcos_resnet50_fpn` backbone, or `create_feature_extractor` + `torchvision.ops.FeaturePyramidNetwork` |
| `FCOSPredictionNetwork`, `FCOS.forward`, loss code | `torchvision.models.detection.FCOS` (focal loss, GIoU loss, centerness BCE) |
| `fcos_match_locations_to_gt` | built into torchvision FCOS (center sampling + per-level scale ranges) |
| `nms`, `class_spec_nms` | `torchvision.ops.nms` / `batched_nms` (reference NMS kept for understanding) |
| `train_detector` helper | plain PyTorch loop: SGD, warmup + cosine LR, AMP, grad clipping |
| Cartucho mAP repo | `torchmetrics.detection.MeanAveragePrecision` |
| `eecs598.utils.detection_visualizer` | `torchvision.utils.draw_bounding_boxes` |

No local `.py` files are needed; the notebook runs on its own (Colab or local, GPU strongly recommended).

## Setup

In [ ]:
# torchvision >= 0.16 is needed for transforms.v2 / tv_tensors.
%pip install -q "torchvision>=0.16" torchmetrics pycocotools

In [ ]:
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchmetrics.detection import MeanAveragePrecision
from torchvision import tv_tensors
from torchvision.datasets import VOCDetection
from torchvision.ops import batched_nms, box_iou, nms, sigmoid_focal_loss
from torchvision.transforms import v2 as T
from torchvision.utils import draw_bounding_boxes

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 8)

print(f"torch {torch.__version__} | torchvision {torchvision.__version__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"  # mixed precision only on CUDA
print("device:", DEVICE, "| AMP:", USE_AMP)


def seed_everything(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(0)

### Configuration

`BACKBONE = "resnet50"` fine-tunes torchvision's COCO-pretrained FCOS and only re-initialises the
classification head for the 20 VOC classes; this is the standard recipe and gives the best accuracy for the
compute. `BACKBONE = "regnet_x_400mf"` rebuilds the small RegNet + FPN model from the original assignment out of
library parts (ImageNet-pretrained trunk, randomly initialised FPN and head). It is much lighter but needs a
longer schedule.

On Colab, point `DATA_ROOT` at Google Drive (after `drive.mount`) so the dataset survives runtime restarts.

In [ ]:
DATA_ROOT = Path("data")          # e.g. Path("/content/drive/MyDrive/datasets") on Colab
BACKBONE = "resnet50"             # "resnet50" (COCO-pretrained FCOS) | "regnet_x_400mf" (light)
MIN_SIZE, MAX_SIZE = 512, 853     # the model resizes the shorter side to MIN_SIZE (capped at MAX_SIZE)
BATCH_SIZE = 8                    # lower this if you run out of GPU memory
NUM_EPOCHS = 12                   # ~3.7k iterations on VOC07 train; use 30+ for the regnet backbone
LR = 0.005                        # for batch size 8; scale linearly with batch size
WEIGHT_DECAY = 1e-4
WARMUP_ITERS = 500
NUM_WORKERS = min(4, os.cpu_count() or 1)
CKPT_PATH = Path("checkpoints") / f"fcos_{BACKBONE}_voc07.pt"

## Data: PASCAL VOC 2007

`torchvision.datasets.VOCDetection` parses the XML annotations; a thin wrapper turns them into the format every
torchvision detector expects:

- **image**: `float32` tensor `(3, H, W)` in `[0, 1]`, any size. The model's own `GeneralizedRCNNTransform` handles
  resizing and ImageNet normalisation, so there is no manual `Normalize` / un-normalise step.
- **target**: `{"boxes": BoundingBoxes (N, 4) XYXY in absolute pixels, "labels": int64 (N,)}`, with label `0`
  reserved for background, so VOC classes are `1..20`.

Batches are *lists* of images and targets, so images of different sizes and different box counts need no padding
(this replaces the old `(B, 40, 5)` tensor padded with `-1`). Boxes flagged *difficult* are dropped, as in the
standard VOC protocol. `transforms.v2` augments the image and its boxes together.

If the official VOC server is slow or down, download the tar manually and extract it under `DATA_ROOT`:

```bash
wget https://web.eecs.umich.edu/~justincj/data/VOCtrainval_06-Nov-2007.tar
tar xf VOCtrainval_06-Nov-2007.tar -C data/   # gives data/VOCdevkit/VOC2007/...
```

In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor",
]
IDX_TO_NAME = ["__background__"] + VOC_CLASSES
CLASS_TO_IDX = {name: i for i, name in enumerate(IDX_TO_NAME)}
NUM_CLASSES = len(IDX_TO_NAME)  # 21: torchvision counts the background slot


class VOCDetectionV2(Dataset):
    """VOC2007 -> (image, {"boxes": BoundingBoxes[XYXY], "labels": int64}) for torchvision detectors."""

    def __init__(self, root, image_set, transforms=None, download=False):
        self.voc = VOCDetection(root, year="2007", image_set=image_set, download=download)
        self.transforms = transforms

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, ann = self.voc[idx]
        objects = [o for o in ann["annotation"]["object"] if o["difficult"] == "0"]
        boxes = torch.tensor(
            [[float(o["bndbox"][k]) for k in ("xmin", "ymin", "xmax", "ymax")] for o in objects],
            dtype=torch.float32,
        ).reshape(-1, 4)
        boxes[:, :2] -= 1  # VOC pixel indices are 1-based
        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(image.height, image.width)),
            "labels": torch.tensor([CLASS_TO_IDX[o["name"]] for o in objects], dtype=torch.int64),
        }
        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target


train_tf = T.Compose([
    T.ToImage(),
    T.RandomPhotometricDistort(p=0.5),
    T.RandomHorizontalFlip(p=0.5),
    T.ToDtype(torch.float32, scale=True),
    T.SanitizeBoundingBoxes(),  # drops boxes that became degenerate
])
eval_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

download = not (DATA_ROOT / "VOCdevkit" / "VOC2007").exists()
train_ds = VOCDetectionV2(DATA_ROOT, "train", train_tf, download=download)
val_ds = VOCDetectionV2(DATA_ROOT, "val", eval_tf)
print(f"train: {len(train_ds)} images | val: {len(val_ds)} images")


def collate_fn(batch):
    return tuple(zip(*batch))  # (images, targets) as tuples, no padding


loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

images, targets = next(iter(train_loader))
print(f"batch: {len(images)} images, first image {tuple(images[0].shape)}")
print("first target:", {k: (type(v).__name__, tuple(v.shape)) for k, v in targets[0].items()})

### Visualise a few training samples

In [ ]:
def to_uint8(image):
    return (image.detach().cpu().clamp(0, 1) * 255).to(torch.uint8)


def show(image, target=None, pred=None, score_thresh=0.5, ax=None, title=None):
    """Ground truth in green, predictions (score >= score_thresh) in red."""
    ax = ax or plt.gca()
    img = to_uint8(image)
    if target is not None and len(target["boxes"]):
        names = [IDX_TO_NAME[l] for l in target["labels"].tolist()]
        img = draw_bounding_boxes(img, target["boxes"].cpu(), labels=names, colors="lime", width=2)
    if pred is not None:
        keep = pred["scores"] >= score_thresh
        if keep.any():
            names = [f"{IDX_TO_NAME[l]} {s:.2f}"
                     for l, s in zip(pred["labels"][keep].tolist(), pred["scores"][keep].tolist())]
            img = draw_bounding_boxes(img, pred["boxes"][keep].cpu(), labels=names, colors="red", width=2)
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")
    if title:
        ax.set_title(title)


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, idx in zip(axes, range(3)):
    show(*train_ds[idx], ax=ax, title=f"train[{idx}]")
plt.tight_layout()
plt.show()

## Build the model

Two standard ways to adapt a torchvision detector to a new dataset:

1. **Fine-tune a pretrained detector.** Load COCO weights and swap the class-dependent layer. For FCOS that
   is the classification head; box regression and centerness are class-agnostic, so they keep their COCO weights.
2. **Plug in your own backbone.** Anything that returns an `OrderedDict` of feature maps and has an
   `out_channels` attribute works. `LastLevelP6P7` adds the two extra pyramid levels that FCOS uses (P3 to P7).

In [ ]:
from torchvision.models import RegNet_X_400MF_Weights, regnet_x_400mf
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.ops import FeaturePyramidNetwork, FrozenBatchNorm2d


class RegNetFPN(nn.Module):
    """RegNetX-400MF trunk + torchvision FPN (the backbone from the original assignment, built from library parts).

    torchvision detectors accept any backbone that returns an OrderedDict of feature maps and exposes
    `out_channels`. BatchNorm is frozen because detection batches are too small for good BN statistics.
    """

    def __init__(self, out_channels=128, extra_blocks=None):
        super().__init__()
        trunk = regnet_x_400mf(weights=RegNet_X_400MF_Weights.IMAGENET1K_V2, norm_layer=FrozenBatchNorm2d)
        self.body = create_feature_extractor(
            trunk,
            return_nodes={"trunk_output.block2": "p3", "trunk_output.block3": "p4", "trunk_output.block4": "p5"},
        )
        with torch.no_grad():
            in_channels = [f.shape[1] for f in self.body(torch.zeros(1, 3, 64, 64)).values()]
        self.fpn = FeaturePyramidNetwork(in_channels, out_channels, extra_blocks=extra_blocks)
        self.out_channels = out_channels

    def forward(self, x):
        return self.fpn(self.body(x))

In [ ]:
from torchvision.models.detection import FCOS, FCOS_ResNet50_FPN_Weights, fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.ops.feature_pyramid_network import LastLevelP6P7


def build_fcos(backbone=BACKBONE):
    if backbone == "resnet50":
        model = fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.COCO_V1, min_size=MIN_SIZE, max_size=MAX_SIZE)
        old_head = model.head.classification_head
        model.head.classification_head = FCOSClassificationHead(
            in_channels=model.backbone.out_channels,
            num_anchors=old_head.num_anchors,
            num_classes=NUM_CLASSES,
        )
    elif backbone == "regnet_x_400mf":
        fpn_backbone = RegNetFPN(out_channels=128, extra_blocks=LastLevelP6P7(128, 128))
        model = FCOS(fpn_backbone, num_classes=NUM_CLASSES, min_size=MIN_SIZE, max_size=MAX_SIZE)
    else:
        raise ValueError(f"unknown backbone {backbone!r}")
    return model


model = build_fcos()
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FCOS/{BACKBONE}: {n_params / 1e6:.1f}M params ({n_train / 1e6:.1f}M trainable)")
print("input transform:", model.transform)
print("location sizes per level:", model.anchor_generator.sizes)

### Feature pyramid

Each pyramid level has the same channel count but a different stride. Every spatial location on every level makes
one prediction (class scores, 4 box distances, centerness), which is what *fully convolutional* means here.

In [ ]:
model.eval()
with torch.no_grad():
    feats = model.backbone(torch.randn(1, 3, MIN_SIZE, MIN_SIZE))
total = 0
for name, f in feats.items():
    total += f.shape[-2] * f.shape[-1]
    print(f"{name:>4}: {tuple(f.shape)}   stride {MIN_SIZE // f.shape[-1]}")
print(f"-> {total} locations, i.e. {total} predictions per {MIN_SIZE}x{MIN_SIZE} image")

## Core concept 1: locations, LTRB targets and centerness

FCOS has no anchor boxes. Each feature-map cell `(i, j)` at stride `s` maps to the image point
`((j + 0.5)·s, (i + 0.5)·s)`. A location that falls on a GT box regresses its distances to the four box edges,
normalised by the stride so that the targets have a similar range on every level:

$$l = \frac{x - x_1}{s},\quad t = \frac{y - y_1}{s},\quad r = \frac{x_2 - x}{s},\quad b = \frac{y_2 - y}{s}$$

Locations near the edge of a box produce poor boxes, so FCOS also predicts a **centerness** score that is
multiplied into the class score at test time:

$$\text{centerness} = \sqrt{\frac{\min(l, r)}{\max(l, r)} \cdot \frac{\min(t, b)}{\max(t, b)}}$$

torchvision represents each location as a stride-sized square "anchor" and encodes boxes with `BoxLinearCoder`,
which is exactly this transform. The check below confirms it.

In [ ]:
def fcos_locations(feat_h, feat_w, stride, device=None):
    """(H*W, 2) image-space (x, y) centres of every cell of a stride-`stride` feature map."""
    xs = (torch.arange(feat_w, device=device) + 0.5) * stride
    ys = (torch.arange(feat_h, device=device) + 0.5) * stride
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    return torch.stack([xx, yy], dim=-1).reshape(-1, 2)


def ltrb_encode(points, boxes, stride):
    """Distances from each point to the left/top/right/bottom edge of its box, divided by stride."""
    x, y = points.unbind(-1)
    return torch.stack([x - boxes[:, 0], y - boxes[:, 1], boxes[:, 2] - x, boxes[:, 3] - y], dim=-1) / stride


def ltrb_decode(deltas, points, stride):
    d = deltas * stride
    x, y = points.unbind(-1)
    return torch.stack([x - d[:, 0], y - d[:, 1], x + d[:, 2], y + d[:, 3]], dim=-1)


def centerness(ltrb):
    lr, tb = ltrb[:, [0, 2]], ltrb[:, [1, 3]]
    return torch.sqrt((lr.min(-1).values / lr.max(-1).values) * (tb.min(-1).values / tb.max(-1).values))


# --- checks --------------------------------------------------------------------------------------
from torchvision.models.detection._utils import BoxLinearCoder

print("first p5 locations:", fcos_locations(2, 2, 32).tolist())   # [[16,16],[48,16],[16,48],[48,48]]

boxes = torch.tensor([[10.0, 15, 100, 115], [30, 20, 40, 30], [120, 100, 200, 200]])
points = torch.tensor([[30.0, 40], [32, 29], [125, 150]])
stride = 8

deltas = ltrb_encode(points, boxes, stride)
torch.testing.assert_close(ltrb_decode(deltas, points, stride), boxes)

coder = BoxLinearCoder(normalize_by_size=True)
decode = getattr(coder, "decode_single", None) or coder.decode   # renamed across torchvision versions
anchors = torch.cat([points - stride / 2, points + stride / 2], dim=1)
torch.testing.assert_close(decode(deltas, anchors), boxes)

torch.testing.assert_close(centerness(deltas), torch.tensor([0.3086067, 0.1666667, 0.2581989]))
print("LTRB encoding matches torchvision's BoxLinearCoder, centerness OK")

In [ ]:
# Centerness targets for every stride-16 location inside one GT box of a real image.
image, target = val_ds[0]
gt = target["boxes"].as_subclass(torch.Tensor)
box_idx = int((gt[:, 2] - gt[:, 0]).mul(gt[:, 3] - gt[:, 1]).argmax())  # largest box
box, stride = gt[box_idx], 16

H, W = image.shape[-2:]
pts = fcos_locations(math.ceil(H / stride), math.ceil(W / stride), stride)
inside = (pts[:, 0] > box[0]) & (pts[:, 0] < box[2]) & (pts[:, 1] > box[1]) & (pts[:, 1] < box[3])
ctr = centerness(ltrb_encode(pts[inside], box.expand(int(inside.sum()), 4), stride))

fig, ax = plt.subplots(figsize=(10, 8))
show(image, {"boxes": box[None], "labels": target["labels"][box_idx : box_idx + 1]}, ax=ax,
     title=f"stride-{stride} locations; colour = centerness target")
ax.scatter(pts[~inside, 0], pts[~inside, 1], s=3, c="white", alpha=0.6)
sc = ax.scatter(pts[inside, 0], pts[inside, 1], s=30, c=ctr, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, fraction=0.035)
plt.show()

### How torchvision assigns targets (replaces `fcos_match_locations_to_gt`)

Inside `FCOS.compute_loss`, for each image:

1. **Center sampling.** A location is a positive candidate for a GT box only if it lies inside the box *and* within
   `center_sampling_radius × stride` (default 1.5) of the box centre. The original FCOS used the whole box; center
   sampling is the improved version from the FCOS follow-up work.
2. **Scale ranges per level.** The largest of the location's `(l, t, r, b)` distances in pixels must fall in that
   level's range: `[0, 64)` for P3, `[64, 128)` for P4, `[128, 256)` for P5, `[256, 512)` for P6 and `[512, ∞)` for P7.
   Small objects are learned on fine levels and large ones on coarse levels.
3. **Ambiguity.** If several boxes still match a location, it takes the box with the smallest area.
   Everything unmatched is background.

The losses are then sigmoid focal loss for classification (normalised by the number of positives), **GIoU loss**
for box regression (the original assignment used L1), and BCE for centerness, the last two on positives only.

## Core concept 2: focal loss

With roughly 5k locations per image and only a handful of positives, plain cross-entropy is dominated by easy
background locations. Focal loss multiplies the per-element BCE by $(1 - p_t)^\gamma$, which drives the loss of
well-classified examples towards zero, and by a class-balancing factor $\alpha_t$:

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

In [ ]:
def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    p = logits.sigmoid()
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * (1 - p_t) ** gamma * ce


logits = torch.randn(4, 1000, NUM_CLASSES)
targets = (torch.rand_like(logits) > 0.99).float()
torch.testing.assert_close(focal_loss(logits, targets), sigmoid_focal_loss(logits, targets))
print("focal loss matches torchvision.ops.sigmoid_focal_loss")

p_t = torch.linspace(0.01, 0.99, 200)
for gamma in [0, 1, 2, 5]:
    label = f"γ = {gamma}" + (" (cross-entropy)" if gamma == 0 else "")
    plt.plot(p_t, focal_loss(torch.logit(p_t), torch.ones_like(p_t), alpha=1.0, gamma=gamma), label=label)
plt.xlabel("probability of the correct class, $p_t$")
plt.ylabel("loss")
plt.title("Focal loss down-weights easy examples")
plt.legend()
plt.show()

## Training and evaluation utilities

A plain PyTorch loop: torchvision detectors return a **loss dict** in `train()` mode and a **list of prediction
dicts** in `eval()` mode, so the loop only sums the losses and calls `backward()`. Standard detection recipe:
SGD with momentum, linear warmup then cosine decay, AMP, gradient clipping.

Evaluation uses `torchmetrics.detection.MeanAveragePrecision` (COCO-style, pycocotools backend) twice:
once at IoU 0.5 only (the VOC metric) and once over IoU 0.50:0.95 (the COCO metric). Note that torchmetrics uses
101-point interpolation, so VOC numbers differ slightly from the 11-point VOC2007 devkit.

In [ ]:
def to_device(images, targets, device):
    images = [img.to(device, non_blocking=True) for img in images]
    targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
    return images, targets


def train_detector(model, loader, *, num_epochs=None, max_iters=None, lr=LR, weight_decay=WEIGHT_DECAY,
                   warmup_iters=WARMUP_ITERS, grad_clip=10.0, device=DEVICE, use_amp=USE_AMP, log_every=50):
    total_iters = max_iters if max_iters is not None else num_epochs * len(loader)
    warmup_iters = max(1, min(warmup_iters, total_iters // 2))

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-3, total_iters=warmup_iters),
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_iters - warmup_iters)),
        ],
        milestones=[warmup_iters],
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, it, start = [], 0, time.time()
    model.train()
    while it < total_iters:
        for images, targets in loader:
            images, targets = to_device(images, targets, device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at iter {it}: {loss_dict}")

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            it += 1
            history.append({"loss": loss.item(), **{k: v.item() for k, v in loss_dict.items()},
                            "lr": optimizer.param_groups[0]["lr"]})
            if it == 1 or it % log_every == 0 or it == total_iters:
                h = history[-1]
                parts = " ".join(f"{k}={v:.3f}" for k, v in h.items() if k not in ("loss", "lr"))
                print(f"[{it:>5}/{total_iters}] loss={h['loss']:.3f} | {parts} | "
                      f"lr={h['lr']:.2e} | {time.time() - start:.0f}s")
            if it >= total_iters:
                break
    return history


def plot_history(history, smooth=20):
    fig, ax = plt.subplots(figsize=(10, 4))
    for key in [k for k in history[0] if k != "lr"]:
        values = torch.tensor([h[key] for h in history])
        if len(values) >= smooth:
            values = values.unfold(0, smooth, 1).mean(1)
        ax.plot(values, label=key)
    ax.set_xlabel("iteration")
    ax.set_ylabel("loss")
    ax.set_title(f"Training losses (moving average over {smooth} iters)")
    ax.legend()
    plt.show()


@torch.no_grad()
def evaluate(model, loader, device=DEVICE, max_batches=None):
    model.eval()
    voc_map = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5], class_metrics=True)
    coco_map = MeanAveragePrecision(iou_type="bbox")
    for i, (images, targets) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        outputs = model([img.to(device) for img in images])
        preds = [{k: v.cpu() for k, v in o.items()} for o in outputs]
        gts = [{"boxes": t["boxes"].as_subclass(torch.Tensor), "labels": t["labels"]} for t in targets]
        voc_map.update(preds, gts)
        coco_map.update(preds, gts)
    return voc_map.compute(), coco_map.compute()


def report(voc_res, coco_res):
    print(f"VOC  mAP@0.5       : {voc_res['map'].item():.3f}")
    print(f"COCO mAP@[.50:.95] : {coco_res['map'].item():.3f} "
          f"(AP50 {coco_res['map_50'].item():.3f}, AP75 {coco_res['map_75'].item():.3f}, "
          f"AP_small {coco_res['map_small'].item():.3f})")
    per_class = sorted(
        zip([IDX_TO_NAME[c] for c in voc_res["classes"].tolist()], voc_res["map_per_class"].tolist()),
        key=lambda kv: kv[1],
    )
    names, aps = zip(*per_class)
    plt.figure(figsize=(8, 7))
    plt.barh(names, aps)
    plt.xlabel("AP@0.5")
    plt.title("Per-class AP on VOC2007 val")
    plt.tight_layout()
    plt.show()

## The torchvision detection API contract

One forward pass in each mode shows everything the training loop and evaluator rely on.

In [ ]:
model = model.to(DEVICE)
images, targets = to_device(*next(iter(train_loader)), DEVICE)

model.train()
loss_dict = model(images, targets)
print("train mode ->", {k: round(v.item(), 4) for k, v in loss_dict.items()})

model.eval()
with torch.no_grad():
    outputs = model(images[:2])
print("eval mode  ->", [{k: tuple(v.shape) for k, v in o.items()} for o in outputs])

## Sanity check: overfit 16 images

Before spending GPU time on the full run, make sure the loss collapses on a tiny subset (no augmentation).
If it does not, something is wrong with the data or the model setup.

In [ ]:
seed_everything(0)
overfit_ds = Subset(VOCDetectionV2(DATA_ROOT, "train", eval_tf), range(16))
overfit_loader = DataLoader(overfit_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)

sanity_model = build_fcos().to(DEVICE)
sanity_history = train_detector(sanity_model, overfit_loader, max_iters=300, warmup_iters=50, log_every=50)
plot_history(sanity_history)

sanity_model.eval()
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
with torch.no_grad():
    for ax, (img, tgt) in zip(axes, [overfit_ds[i] for i in range(3)]):
        show(img, tgt, sanity_model([img.to(DEVICE)])[0], ax=ax)
plt.tight_layout()
plt.show()

del sanity_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Train on VOC2007 train

With the defaults (ResNet-50, 512 px, batch 8, AMP) this runs in well under an hour on a Colab T4.

In [ ]:
seed_everything(0)
model = build_fcos().to(DEVICE)
history = train_detector(model, train_loader, num_epochs=NUM_EPOCHS, log_every=100)
plot_history(history)

CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), CKPT_PATH)
print("saved", CKPT_PATH)

In [ ]:
# Optional: reload the trained weights (e.g. in a fresh runtime) instead of re-training.
model = build_fcos()
model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()

## Core concept 3: non-maximum suppression

At test time torchvision FCOS keeps the top `topk_candidates` locations per level whose score
$\sqrt{\sigma(\text{cls}) \cdot \sigma(\text{centerness})}$ exceeds `score_thresh`, decodes their boxes, and runs
**class-aware NMS** with `batched_nms`.

Greedy NMS: take the highest-scoring box, drop every remaining box whose IoU with it exceeds the threshold, repeat.
Class-aware NMS needs no loop over classes: shift each box by `class_id × (max_coordinate + 1)` so that boxes of
different classes can never overlap, then run ordinary NMS once (this is what the old `class_spec_nms` did, and
what `batched_nms` does internally).

In [ ]:
def nms_reference(boxes, scores, iou_threshold):
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i, rest = order[0], order[1:]
        keep.append(i)
        ious = box_iou(boxes[i : i + 1], boxes[rest]).squeeze(0)
        order = rest[ious <= iou_threshold]
    return torch.stack(keep) if keep else torch.empty(0, dtype=torch.long, device=boxes.device)


# Integer-valued boxes keep IoU arithmetic exact, so the comparison is bit-for-bit.
g = torch.Generator().manual_seed(0)
xy = (torch.rand(3000, 2, generator=g) * 400).round()
wh = (torch.rand(3000, 2, generator=g) * 100).round() + 1
boxes = torch.cat([xy, xy + wh], dim=1)
scores = torch.rand(3000, generator=g)
labels = torch.randint(1, NUM_CLASSES, (3000,), generator=g)

t0 = time.perf_counter(); keep_ref = nms_reference(boxes, scores, 0.5); t1 = time.perf_counter()
keep_tv = nms(boxes, scores, 0.5); t2 = time.perf_counter()
assert torch.equal(keep_ref, keep_tv)
print(f"NMS matches torchvision: kept {len(keep_tv)}/{len(boxes)} | "
      f"reference {1e3 * (t1 - t0):.1f} ms vs torchvision {1e3 * (t2 - t1):.1f} ms")

offsets = labels.to(boxes) * (boxes.max() + 1)
assert torch.equal(nms(boxes + offsets[:, None], scores, 0.5), batched_nms(boxes, scores, labels, 0.5))
print("offset trick == batched_nms (class-aware NMS)")

## Inference

`score_thresh`, `nms_thresh`, `detections_per_img` and `topk_candidates` are plain attributes on the model.
A low score threshold (0.05) is standard for mAP evaluation; the plots below only draw boxes scoring at least 0.5.

In [ ]:
model.score_thresh = 0.05
model.nms_thresh = 0.6
model.eval()
idxs = torch.linspace(0, len(val_ds) - 1, steps=6).long().tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
with torch.no_grad():
    for ax, idx in zip(axes.flat, idxs):
        img, tgt = val_ds[idx]
        show(img, tgt, model([img.to(DEVICE)])[0], score_thresh=0.5, ax=ax, title=f"val[{idx}]")
plt.tight_layout()
plt.show()

## Evaluation on VOC2007 val

For reference, the from-scratch RegNet FCOS in the original assignment targeted about 22% mAP@0.5. A
COCO-pretrained ResNet-50 FCOS fine-tuned with these settings should land far above that. Things to try for more:
a larger `MIN_SIZE`, a longer schedule, training on VOC07+12 `trainval`, or stronger augmentation
(`T.RandomIoUCrop`, `T.RandomZoomOut`, multi-scale `min_size`).

In [ ]:
voc_res, coco_res = evaluate(model, val_loader)
report(voc_res, coco_res)